# 09 OPTIMIZE and VACUUM

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Fix the small-files problem with <code>OPTIMIZE</code> (compaction), clean up unreferenced files with <code>VACUUM</code>, understand retention and its effect on time travel, and know the automatic options on Databricks: optimized writes, auto compaction and predictive optimization.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Streaming jobs, frequent small appends and <code>MERGE</code>s create many small files, and every query then spends its time opening files instead of reading data. Meanwhile, old files from updates and deletes pile up in storage and cost money. <code>OPTIMIZE</code> and <code>VACUUM</code> are the two maintenance commands every Delta table needs, and both appear on the exam.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Two different jobs</b><br>
| | <code>OPTIMIZE</code> | <code>VACUUM</code> |<br>|---|---|---|<br>| Purpose | <b>Compact</b> many small files into fewer large ones (target around 1 GB) | <b>Delete</b> data files no longer referenced by the table and older than the retention period |<br>| Changes the data? | No: same rows, new files. It's a commit (<code>OPTIMIZE</code>) | No new version: it removes files on storage |<br>| Affects time travel | No | Yes: versions that needed the deleted files can't be read any more |<br>| Typical schedule | Daily or after big loads (or automatic) | Daily or weekly (or automatic) |<br><br>
Old files remain after <code>OPTIMIZE</code> (the old small files are "removed" in the log but still on storage), so <code>VACUUM</code> is what actually frees the space.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An IoT team appended sensor readings every minute: after a month, the table had 45,000 files averaging 200 KB, and a simple daily report took 9 minutes, mostly listing and opening files. A nightly <code>OPTIMIZE</code> compacted them into about 60 files and the report dropped to 25 seconds. A weekly <code>VACUUM</code> deleted the 45,000 obsolete small files and cut the storage bill by 40%. Today they would just let <b>predictive optimization</b> run both automatically on the managed table.
</div>

## Syntax

```sql
OPTIMIZE t;                                   -- compact the whole table
OPTIMIZE t WHERE event_date >= '2024-11-01';  -- only some partitions (partitioned tables)
OPTIMIZE t ZORDER BY (customer_id);           -- compact + co-locate values (lesson 10)

VACUUM t DRY RUN;                             -- list files that would be deleted
VACUUM t;                                     -- delete unreferenced files older than the retention (7 days)
VACUUM t RETAIN 240 HOURS;                    -- keep 10 days instead

ALTER TABLE t SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true',
                                 'delta.autoOptimize.autoCompact' = 'true');
```

## Setup: create the small-files problem

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a table with 40 tiny appends, as a frequent micro-batch job would.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
About 40 files with an average size of a few KB, and 40 versions.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

table = f"{DB}.sensor_readings"
spark.sql(f"CREATE OR REPLACE TABLE {table} (sensor_id INT, reading DOUBLE, minute INT)")
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'false', 'delta.autoOptimize.optimizeWrite' = 'false')")

for minute in range(40):
    (spark.range(250).coalesce(1)
     .select((F.col("id") % 50).cast("int").alias("sensor_id"),
             (F.rand(seed=minute) * 100).alias("reading"),
             F.lit(minute).alias("minute"))
     .write.mode("append").saveAsTable(table))

def file_stats():
    d = spark.sql(f"DESCRIBE DETAIL {table}").first()
    return d["numFiles"], d["sizeInBytes"]

files, size = file_stats()
print(f"files: {files} | total size: {size / 1024:.1f} KB | average file: {size / files / 1024:.1f} KB")

## 1. Measure the cost of small files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Times a full scan of the 40-file table.<br><br>
<b>Why it matters</b><br>Each file costs a listing, an open and a footer read, whatever its size. Many small files make even simple queries slow, and the overhead grows linearly with the file count.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A row count of 10,000 and a timing. Compare it with the timing after <code>OPTIMIZE</code> in section 2.
</div>

In [0]:
import time

def timed_scan(label):
    start = time.time()
    n = spark.table(table).filter("reading > 50").count()
    print(f"{label:<18} rows>50: {n:<6} {time.time() - start:.2f}s")

timed_scan("before OPTIMIZE")

## 2. OPTIMIZE: compact the files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs <code>OPTIMIZE</code>, shows its metrics, and compares the file count and scan time.<br><br>
<b>Why it matters</b><br><code>OPTIMIZE</code> rewrites small files into larger ones and commits the change as a new version. The data is identical. Queries now open a handful of files. It's idempotent: running it again on a compacted table does almost nothing.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>numFilesRemoved</code> about 40 and <code>numFilesAdded</code> 1 (the data is tiny), then 1 file, and a faster scan.
</div>

In [0]:
result = spark.sql(f"OPTIMIZE {table}")
m = result.first()["metrics"]
print("files removed:", m["numFilesRemoved"], "| files added:", m["numFilesAdded"])

files, size = file_stats()
print(f"files now: {files} | average file: {size / files / 1024:.1f} KB")
timed_scan("after OPTIMIZE")

## 3. The old files are still there

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows that the table history contains the <code>OPTIMIZE</code>, and that old versions still read the old small files.<br><br>
<b>Why it matters</b><br><code>OPTIMIZE</code> marks the small files as removed in the log, but doesn't delete them. Time travel to earlier versions still works, and the storage still holds both copies until <code>VACUUM</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The history ends with <code>OPTIMIZE</code>, and version 5 still returns its 1,250 rows.
</div>

In [0]:
spark.sql(f"DESCRIBE HISTORY {table} LIMIT 3").select("version", "operation", "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles").show()
print("rows at version 5:", spark.read.option("versionAsOf", 5).table(table).count())

## 4. VACUUM: DRY RUN first

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists which files <code>VACUUM</code> would delete with the default 7-day retention, then with zero retention.<br><br>
<b>Why it matters</b><br><code>VACUUM</code> only deletes files that are <b>not referenced by the current version</b> and <b>older than the retention period</b>. Just after <code>OPTIMIZE</code>, nothing is older than 7 days, so the default run deletes nothing. <code>DRY RUN</code> is the safe way to check before deleting.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Default retention: 0 files listed. Zero retention is refused by the safety check unless the table's retention allows it.
</div>

In [0]:
print("files VACUUM would delete (7-day retention):", spark.sql(f"VACUUM {table} DRY RUN").count())

try:
    spark.sql(f"VACUUM {table} RETAIN 0 HOURS DRY RUN").count()
except Exception as e:
    print("RETAIN 0 HOURS refused by the safety check:", getattr(e, "getCondition", lambda: None)() or type(e).__name__)

## 5. VACUUM for real, and what it does to time travel

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
For this demo only, lowers the table's file retention to 0 hours, runs <code>VACUUM</code>, and then tries to read an old version.<br><br>
<b>Why it matters</b><br>This shows the trade-off directly: after <code>VACUUM</code>, storage is freed, but time travel to versions that needed the deleted files fails. Never use 0 hours on production tables: a long-running query or a streaming job may still be reading those files.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
Lowering <code>delta.deletedFileRetentionDuration</code> below 7 days is a demo technique. In production, keep the default (or longer) so concurrent readers and time travel keep working.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A list of deleted files (about 40). The current version still has 10,000 rows. Reading version 5 now fails with a file-not-found style error.
</div>

In [0]:
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
try:
    deleted = spark.sql(f"VACUUM {table} RETAIN 0 HOURS")
    print("VACUUM finished:", deleted.columns, deleted.count(), "row(s) returned")
except Exception as e:
    print("VACUUM with zero retention not allowed here:", getattr(e, "getCondition", lambda: None)() or type(e).__name__)

print("current rows:", spark.table(table).count())
try:
    # Aggregate a real column: Delta can answer a plain count() from the log's statistics without reading any file.
    spark.read.option("versionAsOf", 5).table(table).agg(F.sum("reading")).collect()
    print("version 5 still readable (files were not vacuumed)")
except Exception as e:
    print("version 5 is no longer readable:", type(e).__name__)

## 6. Automatic maintenance on Databricks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the table properties for optimized writes and auto compaction, and explains predictive optimization.<br><br>
<b>Why it matters</b>
<ul><li><b>Optimized writes</b> (<code>delta.autoOptimize.optimizeWrite</code>): Databricks shuffles data before writing so each partition gets fewer, larger files</li><li><b>Auto compaction</b> (<code>delta.autoOptimize.autoCompact</code>): after a write, small files are compacted automatically</li><li><b>Predictive optimization</b> (Unity Catalog managed tables): Databricks decides when to run <code>OPTIMIZE</code>, <code>VACUUM</code> and statistics collection for you, based on how tables are used. It's the recommended default, and it's why managed tables need little manual maintenance</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The two properties set to true on the table.
</div>

In [0]:
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true')")
spark.sql(f"SHOW TBLPROPERTIES {table}").filter("key LIKE 'delta.autoOptimize%' OR key LIKE 'delta.deletedFile%'").show(truncate=False)

In [0]:
# On Databricks you can check whether predictive optimization is active for the schema/table
try:
    display(spark.sql(f"DESCRIBE TABLE EXTENDED {table}").filter("col_name ILIKE '%predictive%'"))
except Exception as e:
    print("not available:", type(e).__name__)

## Under the hood

```
OPTIMIZE t
  1. find small files (per partition), group them into bins of ~1 GB
  2. rewrite each bin into one new file
  3. commit: remove <small files>, add <big files>   (dataChange = false: streams ignore it)

VACUUM t [RETAIN n HOURS]
  1. list all files in the table directory
  2. keep files referenced by the current version + files removed less than n hours ago
  3. delete the rest (unreferenced and old enough), including orphans from failed writes
  4. no new data version (it's recorded as VACUUM START / VACUUM END in the history)
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> `OPTIMIZE` is a commit. `VACUUM` deletes storage files.

`OPTIMIZE` commits are marked `dataChange = false`, so streaming readers of the table don't reprocess the compacted data.

In [0]:
spark.sql(f"DESCRIBE HISTORY {table} LIMIT 5").select("version", "operation").show()

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {table}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>VACUUM</code> with a short retention broke a running job or time travel</b><br>
Files still needed by readers were deleted. Keep the default 7-day retention (or longer) in production.<br><br>
**⛔ Problem: storage keeps growing even after <code>OPTIMIZE</code>**<br><code>OPTIMIZE</code> doesn't delete files. Run <code>VACUUM</code>, or enable predictive optimization.<br><br>
**⛔ Problem: <code>OPTIMIZE</code> conflicts with a concurrent <code>UPDATE</code> or <code>MERGE</code>**<br>Both rewrite the same files. Schedule them apart, or use row-level concurrency on Databricks.<br><br>
**⛔ Problem: <code>RETAIN 0 HOURS</code> is refused**<br>That's the safety check working. Lower the retention only for demos, and understand the risk.<br><br>
<b>⛔ Problem: thousands of tiny files from streaming</b><br>Enable optimized writes and auto compaction, or rely on predictive optimization, plus periodic <code>OPTIMIZE</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the small-files problem and how does Delta solve it?</b><br>
Many small files make queries slow, because of per-file overhead in listing, opening and reading footers, and they waste metadata. Delta solves it with <code>OPTIMIZE</code>, which compacts files into larger ones, and on Databricks with optimized writes, auto compaction and predictive optimization.<br><br>

<b>🎤 2. What is the difference between <code>OPTIMIZE</code> and <code>VACUUM</code>?</b><br>
<code>OPTIMIZE</code> rewrites small files into bigger ones and commits a new version, with the same data. <code>VACUUM</code> permanently deletes files that the current version no longer references and that are older than the retention period, which frees storage but limits time travel.<br><br>

<b>🎤 3. What is the default <code>VACUUM</code> retention and why does it exist?</b><br>
7 days (<code>delta.deletedFileRetentionDuration</code>). It protects concurrent readers, long-running queries and streaming jobs that may still need older files, and it keeps a week of time travel.<br><br>

<b>🎤 4. Does <code>OPTIMIZE</code> affect streaming readers of a table?</b><br>
No. Its commits are marked as not changing data (<code>dataChange = false</code>), so streams don't reprocess the compacted files.<br><br>

<b>🎤 5. What is predictive optimization?</b><br>
A Databricks feature for Unity Catalog managed tables that automatically runs <code>OPTIMIZE</code>, <code>VACUUM</code> and statistics collection when it determines they're beneficial, removing the need to schedule maintenance yourself.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A Delta table receives small appends every minute and queries have become slow. Which command directly improves read performance by compacting files?</b><br>
A. <code>VACUUM</code><br>
B. <code>OPTIMIZE</code><br>
C. <code>RESTORE</code><br>
D. <code>ANALYZE TABLE</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. After running <code>VACUUM</code> with the default settings, which statement is true?</b><br>
A. All previous versions remain available<br>
B. Unreferenced files older than 7 days are deleted, and versions that needed them can no longer be queried<br>
C. The table is compacted into larger files<br>
D. The transaction log is deleted<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. Which Unity Catalog feature automatically runs <code>OPTIMIZE</code> and <code>VACUUM</code> on managed tables?</b><br>
A. Auto Loader<br>
B. Predictive optimization<br>
C. Delta Sharing<br>
D. Change data feed<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a table and write 30 single-row appends to it. Record the number of files</li><li>Run <code>OPTIMIZE</code> and record the metrics and the new file count</li><li>Run <code>VACUUM ... DRY RUN</code> and explain why nothing is listed</li><li>Enable optimized writes and auto compaction on the table and write 10 more appends. How many files are there now?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
t2 = f"{DB}.many_appends"

# 1. 30 single-row appends
spark.sql(f"CREATE OR REPLACE TABLE {t2} (id INT, v DOUBLE)")
spark.sql(f"ALTER TABLE {t2} SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'false', 'delta.autoOptimize.optimizeWrite' = 'false')")
for i in range(30):
    spark.sql(f"INSERT INTO {t2} VALUES ({i}, {i * 1.5})")
print("files after appends:", spark.sql(f"DESCRIBE DETAIL {t2}").first()["numFiles"])

# 2. Compact
print("OPTIMIZE metrics:", spark.sql(f"OPTIMIZE {t2}").first()["metrics"]["numFilesRemoved"], "files removed")
print("files after OPTIMIZE:", spark.sql(f"DESCRIBE DETAIL {t2}").first()["numFiles"])

# 3. Nothing is older than the 7-day retention yet, so DRY RUN lists no files
print("VACUUM DRY RUN would delete:", spark.sql(f"VACUUM {t2} DRY RUN").count(), "files")

# 4. Automatic compaction (effective on Databricks; open-source Delta supports autoCompact too)
spark.sql(f"ALTER TABLE {t2} SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true')")
for i in range(10):
    spark.sql(f"INSERT INTO {t2} VALUES ({100 + i}, 0.0)")
print("files with auto compaction:", spark.sql(f"DESCRIBE DETAIL {t2}").first()["numFiles"])
spark.sql(f"DROP TABLE {t2}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Small files slow down every read. <code>OPTIMIZE</code> compacts them (same data, new version)</li><li><code>OPTIMIZE</code> doesn't free storage. <code>VACUUM</code> deletes unreferenced files older than the retention</li><li>Default retention: 7 days for files (<code>VACUUM</code>), 30 days for the log (history)</li><li><code>VACUUM</code> limits time travel. Never use very short retention in production</li><li>Use <code>DRY RUN</code> before <code>VACUUM</code></li><li>Databricks: optimized writes, auto compaction, and predictive optimization for managed tables</li></ul>
</div>

| Task | Code |
|---|---|
| Compact | `OPTIMIZE t` |
| Compact + co-locate | `OPTIMIZE t ZORDER BY (col)` |
| Preview cleanup | `VACUUM t DRY RUN` |
| Clean up | `VACUUM t` / `VACUUM t RETAIN 168 HOURS` |
| Auto | `delta.autoOptimize.optimizeWrite`, `delta.autoOptimize.autoCompact` |
| File count | `DESCRIBE DETAIL t` → `numFiles` |

## Git commit

```
git add 03_delta_lake/09_optimize_vacuum.ipynb
git commit -m "add 03_09 optimize vacuum notebook"
```